# 01 Boundaries and the service crosswalk

Fetches fire and rescue authority polygons from the Office for National Statistics
feature service, reconciles them against the territories recorded in the incident
panel, and writes `config/service_crosswalk.csv`. It then assigns HadUK-Grid land cells to
territories and writes the label array used by the weather aggregation.

Run after `scripts/convert_raw.py`.

In [ ]:
import pandas as pd

from fdh import boundaries as bd
from fdh import contract
from fdh.config import boundary_layer, load_config, repo_root

config = load_config()
raw_dir = config.paths.resolved("raw_dir")
interim_dir = config.paths.resolved("interim_dir")
raw_dir, interim_dir

## Fetch

The layer named in `boundaries.layer` is served in EPSG:27700, which matches the HadUK-Grid projection, so no
reprojection is needed before the aggregation.

In [ ]:
layer = boundary_layer(config)
collection = bd.fetch_layer(layer.url)
record = bd.save_layer(collection, raw_dir / "boundaries" / layer.filename)
record

## Reconcile

The join is on E code first and normalised name second.

In [ ]:
polygons, code_field, name_field = bd.load_layer(raw_dir / "boundaries" / layer.filename)
territories, by_year = bd.territories_from_panel(interim_dir)

print(f"{len(polygons)} boundary features ({code_field}, {name_field})")
print(f"{len(territories)} incident territories")

In [ ]:
rows = bd.build_crosswalk(territories, polygons)
crosswalk = pd.DataFrame(rows)
crosswalk

## What did not line up

In [ ]:
print("unmatched incident territories:")
print(crosswalk.loc[crosswalk.match_basis == "unmatched"].to_string(index=False) or "  none")

print("\nmatched on name, E codes differ:")
print(crosswalk.loc[crosswalk.match_basis == "normalised_name"].to_string(index=False))

print("\nboundary features with no incident territory:")
for code, name in bd.unmatched_boundaries(rows, polygons):
    print(f"  {code}  {name}")

In [ ]:
crosswalk.loc[crosswalk.name_differs, ["e_code_territory", "frs_territory", "boundary_name"]]

## Coverage by financial year

Authorities entering or leaving the series show here.

In [ ]:
counts = {year: len(codes) for year, codes in sorted(by_year.items())}
everything = set().union(*by_year.values())
missing = {
    year: sorted((code, territories[code]) for code in everything - codes)
    for year, codes in sorted(by_year.items())
    if everything - codes
}
counts, missing

## Write the crosswalk

The `note` column is for decisions taken by hand: which polygon a merged
territory uses, and what happens to any boundary feature with no territory.
Notes already in the file are carried over when it is rewritten.

In [ ]:
out = repo_root() / "config" / "service_crosswalk.csv"
bd.write_crosswalk(rows, out)
print(f"wrote {out}")

## Rural urban classification

The Office for National Statistics Rural Urban Classification 2021 of fire and
rescue areas. It is an attribute table with no geometry covering England and
Wales, so it is fetched as a table rather than a layer and filtered to England.

The join uses FRA25 codes, so Hampshire and Isle of Wight carries E31000048 here
and E31000017 in the incident data, the same mismatch the crosswalk already
handles. The join is therefore on `boundary_code`, not `e_code_territory`. The
boundary layer carries FRA22 codes, so an authority whose code changed between
2022 and 2025 would show below as unclassified rather than be matched by name.

In [ ]:
classification_source = config.sources["ons_fra_classification"][0]
table_path = raw_dir / "boundaries" / classification_source.filename
table_rows = bd.fetch_table(classification_source.url)
table_record = bd.save_table(table_rows, table_path)
table_record

In [ ]:
classification = bd.load_classification(table_path)
england = {code: row for code, row in classification.items() if code.startswith("E")}
print(f"{len(classification)} classified areas, {len(england)} in England")

In [ ]:
ruc = pd.DataFrame(list(england.values()))
classified = crosswalk.merge(
    ruc, how="left", left_on="boundary_code", right_on=contract.CLASSIFICATION_KEY
).sort_values(contract.RURAL_SHARE, ascending=False)
classified[
    [
        "e_code_territory",
        "frs_territory",
        "boundary_code",
        contract.SETTLEMENT_CLASS,
        contract.RURAL_SHARE,
    ]
]

In [ ]:
unclassified = classified.loc[
    classified[contract.CLASSIFICATION_KEY].isna(),
    ["e_code_territory", "frs_territory", "boundary_code", "match_basis"],
]
print("crosswalk rows with no classification:")
print(unclassified.to_string(index=False) if len(unclassified) else "  none")

## Grid cells per territory

Each HadUK-Grid land cell is assigned to the territory whose polygon contains
its centre. The grid is identical across variables and months (notebook 03), so
the assignment is built once from one reference file and applies to the whole
archive.

The matched boundary code is mapped through the crosswalk, so the result carries
the E code recorded in the incident data. Isles of Scilly has its own polygon
and no recorded territory, so `boundaries.merged` sends its cells to Cornwall
and Isles of Scilly. The polygons come from the layer named in
`boundaries.layer`.

In [ ]:
from fdh import grid
from fdh import weather as wx
from fdh.config import require

probe_dir = raw_dir / "weather" / "probe"
held = sorted(probe_dir.glob("*.nc"))
if held:
    reference = held[0]
else:
    variable = require(config.weather.variables, "weather.variables")[0]
    listing = wx.daily_files(
        require(config.weather.base_url, "weather.base_url"),
        variable,
        require(config.weather.version_directory, "weather.version_directory"),
        config.weather.window_start,
        config.weather.release_covers_to,
    )
    reference = wx.download(listing[0], probe_dir)
print(f"reference file: {reference.name}")

### Guards

Each of these stops the build when it fails: validity constant within the
reference file, 1000 m spacing in both directions, boundary coordinates in
British National Grid metres, and at least one cell for every territory in the
crosswalk.

In [ ]:
fill = require(config.weather.fill_value, "weather.fill_value")
eastings, northings, land = grid.read_grid(reference, fill)
print(f"PASS  validity constant within the file: {int(land.sum()):,} land cells "
      f"of {land.size:,}, grid {land.shape[0]} by {land.shape[1]}")
print(f"PASS  spacing {grid.SPACING_METRES:.0f} m in both directions")

shapes = grid.load_boundaries(raw_dir / "boundaries" / layer.filename)
west, south, east, north = shapes.total_bounds
print(f"PASS  boundary coordinates in metres: eastings {west:,.0f} to {east:,.0f}, "
      f"northings {south:,.0f} to {north:,.0f}, {len(shapes)} polygons from "
      f"{layer.era}, {shapes.crs}")

lookup = grid.load_crosswalk(out, config.boundaries.merged)
cells = grid.land_cells(eastings, northings, land)
codes, assignment = grid.assign(cells, shapes, lookup)
print(f"PASS  every crosswalk territory receives cells: {assignment['territories']} "
      f"of {len(set(lookup.values()))}")

### Reported, not stopped

HadUK-Grid covers the whole United Kingdom and the boundary layer covers England
only, so most land cells outside every polygon are Scotland, Wales, Northern
Ireland and the Isle of Man. The rest are coastal cells whose centre falls
outside a polygon clipped to mean high water.

In [ ]:
print(f"land cells:                     {assignment['land_cells']:,}")
print(f"assigned to a territory:        {assignment['assigned']:,}")
print(f"outside every polygon:          {assignment['outside_every_polygon']:,}")
print(f"boundary codes with no crosswalk entry: {assignment['no_crosswalk_entry'] or 'none'}")

### Cells per territory

In [ ]:
summary = grid.summarise(cells, codes)
per_territory = (
    pd.DataFrame(summary)
    .merge(crosswalk[["e_code_territory", "frs_territory"]], on="e_code_territory")
    .sort_values("cells", ascending=False)
    .reset_index(drop=True)
)
print(per_territory[["e_code_territory", "frs_territory", "cells"]].to_string())

### Cells against polygon area

A cell is one square kilometre, so the cells assigned to a territory should
come close to the area of its polygons in square kilometres, the Isles of
Scilly polygon counted with Cornwall. The area is computed from the polygons
themselves. A ratio far from one would mean cells assigned to the wrong polygon
or coordinates in other units; a small shortfall is expected on coasts, where a
cell centre can fall outside a polygon clipped to mean high water.

In [ ]:
square_km = shapes.geometry.area.groupby(shapes.boundary_code.map(lookup)).sum() / 1e6
areas = per_territory.set_index("e_code_territory")[["frs_territory", "cells"]].assign(
    polygon_km2=square_km.round(0)
)
areas["cells_per_km2"] = (areas.cells / areas.polygon_km2).round(3)
print(areas.sort_values("cells_per_km2").to_string())
print(f"\nall territories: {areas.cells.sum():,} cells against "
      f"{areas.polygon_km2.sum():,.0f} square kilometres")

### Write the label array

`grid_labels.npy` holds a service index per grid cell, -1 where no service is
assigned. `grid_labels_index.json` maps each index to its E code and records
what the mask was built from: the boundary layer and the hash of its file, the
hash of the crosswalk, and the merges declared in config. The weather panel
build stops if any of those differs from the current configuration. The
per-territory counts go to the reports directory.

In [ ]:
labels, index = grid.label_array(land.shape, cells, codes)
built_from = grid.mask_basis(
    layer.era, raw_dir / "boundaries" / layer.filename, out, config.boundaries.merged
)
array_path, index_path = grid.save_labels(
    labels,
    index,
    eastings,
    northings,
    reference,
    config.paths.resolved("processed_dir"),
    built_from,
)
counts_path = config.paths.resolved("reports_dir") / "grid_cells_per_territory.csv"
grid.write_summary(summary, counts_path)
print(f"wrote {array_path}")
print(f"wrote {index_path}")
print(f"wrote {counts_path}")
built_from